In [18]:
import os
import logging
import numpy as np
import torch
from datetime import datetime
from types import SimpleNamespace

from open_clip import create_model_and_transforms, get_tokenizer, create_loss
from open_clip_train.data import get_data
from open_clip_train.distributed import is_master, init_distributed_device
from open_clip_train.logger import setup_logging
from open_clip_train.scheduler import cosine_lr
from open_clip_train.train import train_one_epoch, evaluate

# Manually define arguments (instead of parse_args)
args = SimpleNamespace(
    batch_size=2,
    workers=4,
    report_to="tensorboard",
    save_frequency=1,
    logs="logs",
    dataset_type="csv",
    csv_separator=",",
    train_data="C:/Users/Alireza Vaezi/Desktop/MedCLIP-SAMv2/normal_abnormal.csv",
    csv_img_key="first_frame_path",
    csv_caption_key="description",
    lr=1e-3,
    wd=0.1,
    warmup=1000,
    epochs=32,
    model="hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224",
    dhnnce_loss=True,
    temperature_dhnnce=0.6,
    alpha_dhnnce=0.0,
    beta1_dhnnce=0.15,
    beta2_dhnnce=0.15,
    precision="fp32",
    distributed=False,
    wandb=False,
    tensorboard=True,
    log_local=False,
    debug=False,
    name=None,
    device="cuda" if torch.cuda.is_available() else "cpu",
    cache_dir=None,
)

# Set experiment name
if args.name is None:
    date_str = datetime.now().strftime("%Y_%m_%d-%H_%M_%S")
    model_name_safe = args.model.replace("/", "-")
    args.name = f"{date_str}_model_{model_name_safe}_lr_{args.lr}_b_{args.batch_size}_j_{args.workers}"

# Create log directory
log_base_path = os.path.join(args.logs, args.name)
os.makedirs(log_base_path, exist_ok=True)

# Setup logging
args.log_path = os.path.join(log_base_path, "out.log")
setup_logging(args.log_path, logging.INFO)

# Set checkpoint paths
args.checkpoint_path = os.path.join(log_base_path, "checkpoints")
os.makedirs(args.checkpoint_path, exist_ok=True)
args.tensorboard_path = os.path.join(log_base_path, "tensorboard") if args.tensorboard else ""
if args.tensorboard_path:
    os.makedirs(args.tensorboard_path, exist_ok=True)

# Initialize device
device = init_distributed_device(args)

NotADirectoryError: [WinError 267] The directory name is invalid: 'logs\\2025_03_05-20_13_44_model_hf-hub:microsoft-BiomedCLIP-PubMedBERT_256-vit_base_patch16_224_lr_0.001_b_2_j_4'